# Function Calling
The basic idea is to provide a bunch of function definitions to the API and the API will return a JSON with the function name and the arguments to call it with. It is then my responsibility to actually call the function. I can then pass the results back to the LLM with all the previous context and it will return a NUI answer. 

The function definitions are provided in the `tools` section to the API. Each function definition is nothing but a JSONified function signature with the function name, parameter names, and most importantly parameter descriptions. Interestingly enough the signature does not have the return type schema.

If I have provided multiple function definitions, the API might return with multiple function call responses. I can control this behavior with a couple of different switches -
  * Call zero or more: `tool_choice: "auto"`. This is the default behavior.
  * Call one or more: `tool_choice: "required"`.
  * Call zero or one: `parallel_tool_calls: false`.
  * Call exactly one specific: `tool_choice: {"type": "function", "function": {"name": "my_awesome_function"}}`

They recommend setting the strict mode with `strict: true`. This uses the structured output API under the covers and implies that I must not have any optional/default parameters by including all the parameters in the `required` property, and to set `additionalProperties: false`.

There are cases where the user input might not have the exact input that the function needs, the LLM will convert the user provided info into the function inputs, e.g., in one of the examples the `get_weather` function needs the exact lat/long, but the user has only provided the name of a city, the LLM is able to convert this city into lat/long for the function. Even in the simple `get_weather` implementation which just takes in a location string, the model converts my input of "Paris" to "Paris, France" because that is the pattern I have given in my field description.

The functions are injected into the system (developer?) message in a syntax the model has been trained on (so their models are instruction tuned for different use cases?). They keep recommending fine-tuning to improve token efficiency and higher quality performance.

I can use Pydantic to define my function instead of coding up the JSON schema by hand.


In [1]:
from dotenv import load_dotenv
from openai import OpenAI, pydantic_function_tool
import pydantic as pt
import json
from utils import LLM, ChatCompletionMessage, Function
from typing import Any

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country e.g., Seattle, USA",
                    }
                },
                "required": ["location"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [5]:
messages: list[dict[str, str] | ChatCompletionMessage] = [
    {"role": "user", "content": "What is the weather like in Paris today?"}
]

In [6]:
completion = client.chat.completions.create(
    model=LLM.PRE_FAST_MINI,
    messages=messages,  # type: ignore
    tools=tools,  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BNBEb1oxoQDcrVWes0XbbYo56hYam', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_1CZIUrnADRrqPcr3rfRDcRkt', function=Function(arguments='{"location":"Paris, France"}', name='get_weather'), type='function')]))], created=1744864429, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_f7d56a8a2c', usage=CompletionUsage(completion_tokens=17, prompt_tokens=65, total_tokens=82, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

```python
ChatCompletion(
    id='chatcmpl-BN8NdeccLV7VHtkoAbZZGxCEn8d6T', 
    choices=[
        Choice(
            finish_reason='tool_calls', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content=None, 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='call_NR3dbwNzzVzZCLEfpZMgK5Lt', 
                        function=Function(
                            arguments='{"location":"Paris, France"}', 
                            name='get_weather'
                        ), 
                        type='function'
                    )
                ]
            )
        )
    ], 
    created=1744853457, 
    model='gpt-4.1-2025-04-14', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint='fp_beec22d258', 
    usage=CompletionUsage(
        completion_tokens=17, 
        prompt_tokens=65, 
        total_tokens=82, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=0, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [7]:
type(completion.choices[0].message)

openai.types.chat.chat_completion_message.ChatCompletionMessage

In [8]:
def get_weather(location: str) -> int:
    return 15 if location.startswith("Paris") else 25


func_table = {"get_weather": get_weather}


def apply(func: Function) -> Any:
    if func.name in func_table:
        kwargs = json.loads(func.arguments)
        return func_table[func.name](**kwargs)
    return None

In [9]:
assert completion.choices[0].message.tool_calls

# So far messages just had my input, now append the intermediate model output
messages.append(completion.choices[0].message)

# Now call the functions that the model has asked to, and append their results as well
for tool_call in completion.choices[0].message.tool_calls:
    if tool_call.type == "function":
        ret = apply(tool_call.function)
        # Remember to stringify the response when adding it back to the message
        tool_response = {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(ret),
        }
        messages.append(tool_response)

messages

[{'role': 'user', 'content': 'What is the weather like in Paris today?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_1CZIUrnADRrqPcr3rfRDcRkt', function=Function(arguments='{"location":"Paris, France"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_1CZIUrnADRrqPcr3rfRDcRkt',
  'content': '15'}]

In [11]:
# Now call the model again with the results
completion = client.chat.completions.create(model=LLM.PRE_FAST_MINI, messages=messages, tools=tools)  # type: ignore
completion

ChatCompletion(id='chatcmpl-BNBF1gslIhkNE9lg1rDmEMe08Oio7', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current temperature in Paris is 15°C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1744864455, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_f7d56a8a2c', usage=CompletionUsage(completion_tokens=12, prompt_tokens=90, total_tokens=102, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

```python
ChatCompletion(
    id='chatcmpl-BN8PZNMGPtogNzOsLnfIDQxFeVR6W', 
    choices=[
        Choice(
            finish_reason='stop', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content='The current temperature in Paris today is 15°C. If you need more details about the weather (such as conditions, wind, or forecast), let me know!', 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=None
            )
        )
    ], 
    created=1744853577, 
    model='gpt-4.1-2025-04-14', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint='fp_beec22d258', 
    usage=CompletionUsage(
        completion_tokens=35, 
        prompt_tokens=90, 
        total_tokens=125, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=0, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [12]:
completion.choices[0].message.content

'The current temperature in Paris is 15°C.'

I can also use Pydantic to specify the function definition. In the below example I'll demo that as well as the ability of the LLM to convert user provided info into specific function arguments.

In their documentation example, they model the entire function with a `BaseModel`. That seems semantically confusing to me, so I am modeling the "request" to the function as a `BaseModel`.

In [13]:
class GetWeatherRequest(pt.BaseModel):
    """
    Get the current temperature for provided coordinates in celsius.
    """

    latitude: float = pt.Field(description="Latitude of the location.")
    longitude: float = pt.Field(description="Longitude of the location.")

In [14]:
tool = pydantic_function_tool(GetWeatherRequest)

In [15]:
tool

{'type': 'function',
 'function': {'name': 'GetWeatherRequest',
  'strict': True,
  'parameters': {'description': 'Get the current temperature for provided coordinates in celsius.',
   'properties': {'latitude': {'description': 'Latitude of the location.',
     'title': 'Latitude',
     'type': 'number'},
    'longitude': {'description': 'Longitude of the location.',
     'title': 'Longitude',
     'type': 'number'}},
   'required': ['latitude', 'longitude'],
   'title': 'GetWeatherRequest',
   'type': 'object',
   'additionalProperties': False},
  'description': '\nGet the current temperature for provided coordinates in celsius.\n'}}

In [16]:
tool.keys()

dict_keys(['type', 'function'])

In [17]:
tool["function"].keys()

dict_keys(['name', 'strict', 'parameters', 'description'])

In [18]:
tool["function"]["description"]  # type: ignore

'\nGet the current temperature for provided coordinates in celsius.\n'

In [19]:
messages: list[dict[str, str] | ChatCompletionMessage] = [
    {"role": "user", "content": "What is the weather like in Kolkata today?"}
]
tools = [tool]
completion = client.chat.completions.create(
    model=LLM.PRE_FAST_MINI,
    messages=messages,  # type: ignore
    tools=tools,  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BNBFJEYl4z9yr0qbrdLHxHpM5y2Ov', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_Gzg0zDMC2JaV7g3EfhmisStr', function=Function(arguments='{"latitude":22.5726,"longitude":88.3639}', name='GetWeatherRequest'), type='function')]))], created=1744864473, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_f7d56a8a2c', usage=CompletionUsage(completion_tokens=26, prompt_tokens=96, total_tokens=122, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

```python
ChatCompletion(
    id='chatcmpl-BN8bSKWIO2jpQFUqqo9XTUHvmq0jD', 
    choices=[
        Choice(
            finish_reason='tool_calls', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content=None, 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='call_OUwB5FKWo5jEXmCDUNthYhBM', 
                        function=Function(
                            arguments='{"latitude":22.5726,"longitude":88.3639}', 
                            name='GetWeatherRequest'
                        ), 
                        type='function'
                    )
                ]
            )
        )
    ], 
    created=1744854314, 
    model='gpt-4.1-2025-04-14', 
    object='chat.completion', 
    service_tier='default', 
    system_fingerprint='fp_b38e740b47', 
    usage=CompletionUsage(
        completion_tokens=26, 
        prompt_tokens=96, 
        total_tokens=122, 
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0, 
            audio_tokens=0, 
            reasoning_tokens=0, 
            rejected_prediction_tokens=0
        ), 
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)
```

In [20]:
import requests


def get_weather(latitude: float, longitude: float) -> int:
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]["temperature_2m"]

In [21]:
print(completion.choices[0].message.tool_calls[0].function.name)  # type: ignore
kwargs = json.loads(completion.choices[0].message.tool_calls[0].function.arguments)  # type: ignore
kwargs

GetWeatherRequest


{'latitude': 22.5726, 'longitude': 88.3639}

In [22]:
ret = get_weather(**kwargs)
ret

33.2

In [23]:
messages.append(completion.choices[0].message)
tool_response = {
    "role": "tool",
    "tool_call_id": completion.choices[0].message.tool_calls[0].id,  # type: ignore
    "content": str(ret),
}
messages.append(tool_response)
messages

[{'role': 'user', 'content': 'What is the weather like in Kolkata today?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_Gzg0zDMC2JaV7g3EfhmisStr', function=Function(arguments='{"latitude":22.5726,"longitude":88.3639}', name='GetWeatherRequest'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_Gzg0zDMC2JaV7g3EfhmisStr',
  'content': '33.2'}]

In [25]:
completion = client.chat.completions.create(model=LLM.PRE_FAST_MINI, messages=messages, tools=tools)  # type: ignore
completion

ChatCompletion(id='chatcmpl-BNBFl4kHMJEPAgTeJNA8bfEG43DTe', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current temperature in Kolkata today is 33.2°C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1744864501, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_f7d56a8a2c', usage=CompletionUsage(completion_tokens=15, prompt_tokens=133, total_tokens=148, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [26]:
completion.choices[0].message.content

'The current temperature in Kolkata today is 33.2°C.'

In [27]:
class SendEMailReqeust(pt.BaseModel):
    """Send an email to a given recipient with a subject and message."""

    to: str = pt.Field(description="The recipient's email address.")
    subject: str = pt.Field(description="Email subject line.")
    body: str = pt.Field(description="Body of the email message.")

In [28]:
messages: list[dict[str, str] | ChatCompletionMessage] = [
    {
        "role": "user",
        "content": "Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.",
    }
]
tools = [
    pydantic_function_tool(SendEMailReqeust),
    pydantic_function_tool(GetWeatherRequest),
]

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'SendEMailReqeust',
   'strict': True,
   'parameters': {'description': 'Send an email to a given recipient with a subject and message.',
    'properties': {'to': {'description': "The recipient's email address.",
      'title': 'To',
      'type': 'string'},
     'subject': {'description': 'Email subject line.',
      'title': 'Subject',
      'type': 'string'},
     'body': {'description': 'Body of the email message.',
      'title': 'Body',
      'type': 'string'}},
    'required': ['to', 'subject', 'body'],
    'title': 'SendEMailReqeust',
    'type': 'object',
    'additionalProperties': False},
   'description': 'Send an email to a given recipient with a subject and message.'}},
 {'type': 'function',
  'function': {'name': 'GetWeatherRequest',
   'strict': True,
   'parameters': {'description': 'Get the current temperature for provided coordinates in celsius.',
    'properties': {'latitude': {'description': 'Latitude of the location.',


In [30]:
completion = client.chat.completions.create(
    model=LLM.PRE_FAST_MINI,
    messages=messages,  # type: ignore
    tools=tools,  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BNBG18GBpCFapyz1qsb84UbBkF8uh', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_OeFFmNaKA2FgGHebbXNADUNE', function=Function(arguments='{"latitude":22.5726,"longitude":88.3639}', name='GetWeatherRequest'), type='function')]))], created=1744864517, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_0392822090', usage=CompletionUsage(completion_tokens=26, prompt_tokens=199, total_tokens=225, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

```python
ChatCompletion(
    id='chatcmpl-BN8sVa23LY1naJNpC4WJjqMW5xEi7', 
    choices=[
        Choice(
            finish_reason='tool_calls', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content=None, 
                refusal=None, 
                role='assistant',
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='call_4MNecEHf1KdiNdvvZ8VR7KWP', 
                        function=Function(
                            arguments='{"latitude":22.5726,"longitude":88.3639}', 
                            name='GetWeatherRequest'
                        ), 
                        type='function'
                    )
                ]
            )
        )
    ], 
    created=1744855371, model='gpt-4.1-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_b38e740b47', usage=CompletionUsage(completion_tokens=26, prompt_tokens=199, total_tokens=225, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
```

In [31]:
messages

[{'role': 'user',
  'content': 'Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.'}]

In [32]:
messages.append(completion.choices[0].message)
tool_response = {
    "role": "tool",
    "tool_call_id": completion.choices[0].message.tool_calls[0].id,  # type: ignore
    "content": str(28),
}
messages.append(tool_response)
messages

[{'role': 'user',
  'content': 'Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_OeFFmNaKA2FgGHebbXNADUNE', function=Function(arguments='{"latitude":22.5726,"longitude":88.3639}', name='GetWeatherRequest'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_OeFFmNaKA2FgGHebbXNADUNE',
  'content': '28'}]

In [33]:
completion = client.chat.completions.create(model=LLM.PRE_FAST_MINI, messages=messages, tools=tools)  # type: ignore
completion

ChatCompletion(id='chatcmpl-BNBGHBkYJa1zQMcbp5mIqyCbA4AYK', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_TwgIiT9EUb5GAEhGNVUT83DS', function=Function(arguments='{"to":"avilay@gmail.com","subject":"Current Weather Update in Kolkata","body":"Hello Avilay,\\n\\nI hope this email finds you well! \\n\\nThe current temperature in Kolkata is 28°C.\\n\\nBest regards,\\n[Your Name]"}', name='SendEMailReqeust'), type='function')]))], created=1744864533, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_0392822090', usage=CompletionUsage(completion_tokens=71, prompt_tokens=234, total_tokens=305, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_t

```python
ChatCompletion(
    id='chatcmpl-BN8vxBtfa9NRl0b4NFmCaJXzDNik8', 
    choices=[
        Choice(
            finish_reason='tool_calls', 
            index=0, 
            logprobs=None, 
            message=ChatCompletionMessage(
                content=None, 
                refusal=None, 
                role='assistant', 
                annotations=[], 
                audio=None, 
                function_call=None, 
                tool_calls=[
                    ChatCompletionMessageToolCall(
                        id='call_Ao5P7Hh3cY05RyHWJaIP9YSA', 
                        function=Function(
                            arguments='{"to":"avilay@gmail.com","subject":"Weather Update for Kolkata","body":"Hi Avilay,\\n\\nJust wanted to let you know that the current temperature in Kolkata is 28°C.\\n\\nBest regards,"}', 
                            name='SendEMailReqeust'
                        ), 
                        type='function'
                    )
                ]
            )
        )
    ], created=1744855585, model='gpt-4.1-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_b38e740b47', usage=CompletionUsage(completion_tokens=61, prompt_tokens=234, total_tokens=295, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
```

In [34]:
print(completion.choices[0].message.tool_calls[0].function.name)  # type: ignore
kwargs = json.loads(completion.choices[0].message.tool_calls[0].function.arguments)  # type: ignore
kwargs

SendEMailReqeust


{'to': 'avilay@gmail.com',
 'subject': 'Current Weather Update in Kolkata',
 'body': 'Hello Avilay,\n\nI hope this email finds you well! \n\nThe current temperature in Kolkata is 28°C.\n\nBest regards,\n[Your Name]'}

In [35]:
messages.append(completion.choices[0].message)
tool_response = {
    "role": "tool",
    "tool_call_id": completion.choices[0].message.tool_calls[0].id,  # type: ignore
    "content": json.dumps({"status_code": 200, "status": "ok"}),
}
messages.append(tool_response)
messages

[{'role': 'user',
  'content': 'Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_OeFFmNaKA2FgGHebbXNADUNE', function=Function(arguments='{"latitude":22.5726,"longitude":88.3639}', name='GetWeatherRequest'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_OeFFmNaKA2FgGHebbXNADUNE',
  'content': '28'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_TwgIiT9EUb5GAEhGNVUT83DS', function=Function(arguments='{"to":"avilay@gmail.com","subject":"Current Weather Update in Kolkata","body":"Hello Avilay,\\n\\nI hope this email finds you well! \\n\\nThe current temperature in Kolkata is 28°C.\\n\\nBest regards,\\n[Your Name]"}', name='SendEMai

In [36]:
completion = client.chat.completions.create(
    model=LLM.PRE_FAST_MINI,
    messages=messages,  # type: ignore
    tools=tools,  # type: ignore
)
completion

ChatCompletion(id='chatcmpl-BNBGa7xO5kdMCZEbMwpI0En3uEeWZ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I've sent an email to Avilay informing him about the current weather in Kolkata, which is 28°C.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1744864552, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_0392822090', usage=CompletionUsage(completion_tokens=25, prompt_tokens=329, total_tokens=354, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [37]:
completion.choices[0].message.content

"I've sent an email to Avilay informing him about the current weather in Kolkata, which is 28°C."